In [4]:
df.columns

Index(['is_fraud', 'transaction_id', 'user_id', 'device_id',
       'transaction_timestamp', 'user_category_affinity', 'transaction_amount',
       'txn_minute', 'txn_day', 'account_balance', 'txn_day_of_week',
       'balance_utilization', 'txn_hour', 'user_transaction_count',
       'txn_month', 'is_balance_missing', 'merchant_location_lon',
       'device_type_web', 'user_location_lat', 'pay_Wallet'],
      dtype='object')

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# =====================================================================
# 1. LOAD YOUR REAL DATASET
# =====================================================================
# Put the name of your actual 50-feature CSV file here
df = pd.read_csv('/content/synthetic_dataset_with_fraud.csv')

# =====================================================================
# 2. PREPARE THE DATA FOR GRADING
# =====================================================================
# We drop the IDs and the answer key ('is_fraud').
# CRITICAL: We do NOT drop the 'pattern_' columns. We want the AI to grade them!
cols_to_drop = ['is_fraud', 'transaction_id', 'user_id', 'device_id', 'transaction_timestamp', 'Unnamed: 0']

# Create our pool of candidate features (X) and our target (y)
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
X = X.select_dtypes(include=[np.number]) # Ensure we only pass numbers to the AI
y = df['is_fraud']

# =====================================================================
# 3. GRADE THE FEATURES (FEATURE IMPORTANCE)
# =====================================================================
print("🌲 AI is grading all features to find the best 15...")
rf_grader = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_grader.fit(X, y)

# Create a scorecard DataFrame
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_grader.feature_importances_
}).sort_values(by='Importance', ascending=False)

# =====================================================================
# 4. SELECT THE TOP 15 AND BUILD THE NEW DATASET
# =====================================================================
# Grab the names of the top 15 features
top_15_features = feature_scores['Feature'].head(15).tolist()

# Print out the winners so you can put them in your hackathon presentation
print("\n🏆 THE TOP 15 MOST POWERFUL FEATURES:")
for i, feat in enumerate(top_15_features, 1):
    print(f"  {i}. {feat}")

# Combine the necessary ID/Target columns with our 15 winning features
columns_to_keep = [col for col in cols_to_drop if col in df.columns] + top_15_features
df_top_15 = df[columns_to_keep]

# =====================================================================
# 5. VERIFY SHAPES & SAVE
# =====================================================================
print("\n=== DATA SHAPES ===")
print(f"Original Data shape:   {df.shape}")
print(f"Reduced Data shape:    {df_top_15.shape}")

# Save this lean dataset to use for your actual model training
df_top_15.to_csv('final_15_feature_dataset.csv', index=False)
print("\n💾 Saved as 'final_15_feature_dataset.csv'")

🌲 AI is grading all features to find the best 15...

🏆 THE TOP 15 MOST POWERFUL FEATURES:
  1. user_category_affinity
  2. transaction_amount
  3. txn_minute
  4. txn_day
  5. account_balance
  6. txn_day_of_week
  7. balance_utilization
  8. txn_hour
  9. user_transaction_count
  10. txn_month
  11. is_balance_missing
  12. merchant_location_lon
  13. device_type_web
  14. user_location_lat
  15. pay_Wallet

=== DATA SHAPES ===
Original Data shape:   (15000, 40)
Reduced Data shape:    (15000, 20)

💾 Saved as 'final_15_feature_dataset.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, log_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings

# Keep the output clean from warnings
warnings.filterwarnings('ignore')

print("=== STAGE 1: LOADING & PREPARING DATA ===")
df = pd.read_csv('/content/final_15_feature_dataset.csv')

# Drop the engineered patterns and IDs so the AI learns from the raw data, not our cheatsheet
cols_to_drop = ['is_fraud', 'transaction_id', 'user_id', 'device_id', 'transaction_timestamp', 'Unnamed: 0']

X = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')
X = X.select_dtypes(include=['int64', 'float64']) # AI only accepts numbers
y = df['is_fraud']

# Calculate ratio to help XGBoost handle imbalanced data (if frauds are rare)
ratio = float(np.sum(y == 0)) / np.sum(y == 1) if np.sum(y == 1) > 0 else 1.0

# Train/Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows.\n")


print("=== STAGE 2: ALGORITHM SHOOTOUT (GRID SEARCH) ===")

# 1. Define the models we want to test
models = {
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio),
    "LightGBM": LGBMClassifier(random_state=42, class_weight='balanced', verbosity=-1)
}

# 2. Define the settings (Grid) we want to test for each model
param_grids = {
    "Random Forest": {
        'n_estimators': [50, 100],
        'max_depth': [None, 10, 20]
    },
    "XGBoost": {
        'n_estimators': [50, 100],
        'max_depth': [3, 6],
        'learning_rate': [0.01, 0.1]
    },
    "LightGBM": {
        'n_estimators': [50, 100],
        'max_depth': [-1, 10],
        'learning_rate': [0.01, 0.1]
    }
}

best_overall_model = None
best_overall_score = -1
best_model_name = ""

# 3. Loop through each model and find the best settings
for name, model in models.items():
    print(f"\n Training {name}...")

    # Run GridSearch (cv=3 means it double-checks its work 3 times)
    grid = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=3, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best {name} Parameters: {grid.best_params_}")

    # Grab the best version of the current model
    best_current_model = grid.best_estimator_

    # Make predictions on BOTH Train and Test data to check for overfitting
    y_train_pred = best_current_model.predict(X_train)
    y_test_pred = best_current_model.predict(X_test)
    y_test_pred_proba = best_current_model.predict_proba(X_test)

    # Calculate Train Accuracy vs Test Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_loss = log_loss(y_test, y_test_pred_proba)

    print(f" {name} -> Train Acc: {train_acc * 100:.2f}% | Test Acc: {test_acc * 100:.2f}% | Log Loss: {test_loss:.4f}")

    # Save the absolute best model overall
    if grid.best_score_ > best_overall_score:
        best_overall_score = grid.best_score_
        best_overall_model = best_current_model
        best_model_name = name


print("\n" + "="*50)
print(f" CHAMPION MODEL: {best_model_name.upper()} ")
print("="*50)

print("\n=== FINAL EVALUATION ON UNSEEN TEST DATA ===")
# Final predictions using the Champion Model
y_train_final = best_overall_model.predict(X_train)
y_test_final = best_overall_model.predict(X_test)
y_test_proba_final = best_overall_model.predict_proba(X_test)

final_train_accuracy = accuracy_score(y_train, y_train_final)
final_test_accuracy = accuracy_score(y_test, y_test_final)
final_loss = log_loss(y_test, y_test_proba_final)

print(f" TRAIN ACCURACY: {final_train_accuracy * 100:.2f}%")
print(f" TEST ACCURACY:  {final_test_accuracy * 100:.2f}%")
print(f" LOG LOSS: {final_loss:.4f} ")

print("\n CONFUSION MATRIX (On Test Data):")
cm = confusion_matrix(y_test, y_test_final)
print("                 Predicted Normal | Predicted Fraud")
print(f"Actual Normal  |       {cm[0][0]}         |       {cm[0][1]}")
print(f"Actual Fraud   |       {cm[1][0]}         |       {cm[1][1]}")

print("\n DETAILED METRICS (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_test_final, zero_division=0))

=== STAGE 1: LOADING & PREPARING DATA ===
Training on 12000 rows, Testing on 3000 rows.

=== STAGE 2: ALGORITHM SHOOTOUT (GRID SEARCH) ===

 Training Random Forest...
Best Random Forest Parameters: {'max_depth': 10, 'n_estimators': 50}
 Random Forest -> Train Acc: 93.35% | Test Acc: 83.93% | Log Loss: 0.5759

 Training XGBoost...
Best XGBoost Parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
 XGBoost -> Train Acc: 89.20% | Test Acc: 76.13% | Log Loss: 0.5382

 Training LightGBM...
Best LightGBM Parameters: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 100}
 LightGBM -> Train Acc: 90.15% | Test Acc: 76.60% | Log Loss: 0.5348

 CHAMPION MODEL: LIGHTGBM 

=== FINAL EVALUATION ON UNSEEN TEST DATA ===
 TRAIN ACCURACY: 90.15%
 TEST ACCURACY:  76.60%
 LOG LOSS: 0.5348 

 CONFUSION MATRIX (On Test Data):
                 Predicted Normal | Predicted Fraud
Actual Normal  |       2234         |       471
Actual Fraud   |       231         |       64

 DETAILED MET

In [5]:
import joblib
import os

# =====================================================================
# 1. DEFINE YOUR SAVE PATH
# =====================================================================
# It is best practice to create a specific folder for your saved models
SAVE_DIR = "saved_models_2"
MODEL_FILENAME = "New_Model.joblib"

# Create the directory if it doesn't exist yet so the code doesn't crash
os.makedirs(SAVE_DIR, exist_ok=True)

# Combine them into the final full path
full_model_path = os.path.join(SAVE_DIR, MODEL_FILENAME)

# =====================================================================
# 2. SAVE THE MODEL (Architecture + Weights)
# =====================================================================
# Assuming your trained movdel variable is named 'rf' or 'model'
# joblib.dump(your_model_variable, file_path)

print(f"💾 Saving model to {full_model_path}...")
joblib.dump(best_overall_model, full_model_path)
print("✅ Model and weights successfully saved!")

# =====================================================================
# 3. HOW TO LOAD IT BACK (For your API or final testing)
# =====================================================================
# When you open a new notebook or build your web app, you just run this:
print("\n🔄 Testing model reload...")
loaded_fraud_model = joblib.load(full_model_path)
print("✅ Model successfully reloaded and ready for predictions!")

💾 Saving model to saved_models_2/New_Model.joblib...
✅ Model and weights successfully saved!

🔄 Testing model reload...
✅ Model successfully reloaded and ready for predictions!


In [6]:
import pandas as pd
import joblib

# =====================================================================
# 1. LOAD MODEL & EXTRACT TRAINED FEATURES
# =====================================================================
model_path = "saved_models_2/New_Model.joblib"
rf_model = joblib.load(model_path)

# The model explicitly remembers the columns it was trained on
trained_features = list(rf_model.feature_names_in_)

# =====================================================================
# 2. LOAD SYNTHETIC DATA & EXTRACT ITS FEATURES
# =====================================================================
df_synthetic = pd.read_csv('synthetic_15000_dataset (2).csv')

# Drop the target and ID columns to isolate just the predictive features
synthetic_features = list(df_synthetic.drop(columns=['is_fraud', 'user_id', 'transaction_id'], errors='ignore').columns)

# =====================================================================
# 3. COMPARE USING SET OPERATIONS
# =====================================================================
set_trained = set(trained_features)
set_synthetic = set(synthetic_features)

missing_from_synthetic = set_trained - set_synthetic
extra_in_synthetic = set_synthetic - set_trained
perfect_matches = set_trained.intersection(set_synthetic)

# =====================================================================
# 4. PRINT DIAGNOSTIC REPORT
# =====================================================================
print("\n" + "="*50)
print("🔍 FEATURE ALIGNMENT DIAGNOSTIC REPORT")
print("="*50)
print(f"Model expects exactly:  {len(trained_features)} features")
print(f"Synthetic data has:     {len(synthetic_features)} features")
print("-" * 50)

print(f"\n✅ PERFECT MATCHES ({len(perfect_matches)} features):")
# These are the columns that are good to go
for col in sorted(list(perfect_matches)):
    print(f"  - {col}")

print(f"\n❌ MISSING FROM SYNTHETIC ({len(missing_from_synthetic)} features):")
# The model NEEDS these to make a prediction, but they aren't in your new file
if not missing_from_synthetic:
    print("  (None! You have all required columns)")
for col in sorted(list(missing_from_synthetic)):
    print(f"  - {col}")

print(f"\n⚠️ EXTRA IN SYNTHETIC ({len(extra_in_synthetic)} features):")
# The model doesn't know what these are and will crash if you feed them to it
if not extra_in_synthetic:
    print("  (None! No extra clutter)")
for col in sorted(list(extra_in_synthetic)):
    print(f"  - {col}")
print("="*50)

FileNotFoundError: [Errno 2] No such file or directory: 'synthetic_15000_dataset (2).csv'